# 🎯 [MOV02_M06] 전술 수송 경로 및 자산 배정 현황 가시화 2D/3D 수송 관제 대시보드 구현
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m06_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M06 |
| **미션 주제** | 전술 수송 경로 및 자산 배정 현황 가시화 2D/3D 수송 관제 대시보드 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 전술 수송 경로 및 자산 배정 현황 가시화 2D/3D 수송 관제 대시보드 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전술 수송 경로 및 자산 배정 현황 가시화 대시보드를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **수송 요청별 자산 배정과 경로를 관제 대시보드로 가시화하고, 입력 검증기와 에러 처리기로 예외 상황에서도 멈추지 않게 한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **요청을 적재량에 맞게 트레일러·트럭에 배정하고 경로 지도, 자산 가동률, 운행 일정(지연 표시), 요청별 비용을 2x2 대시보드로 그린다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/utils/mov02_validator.py 및 에러 핸들러` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 사용자에게 알기 쉬운 에러 메시지 및 조치 가이드 제공

#### 미션 PASS 조건

- 정량 기준: 예외 상황 발생 시 크래시(Crash) 0건, 에러 로그 기록률 100%


- 진행 기준: 예외 주입 테스트 10종 통과 및 감사 로그 파일 생성


- 검증 방법: 비정상 데이터(결측, 특수문자, 대용량) 스트레스 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""수송 요청에 자산(트럭·트레일러)과 위협 회피 경로를 배정해 관제 대시보드(경로 지도·자산 가동률·운행 일정·비용 실적)로 가시화하고, 입력 검증기·에러 처리기로 예외 주입 10종을 크래시 없이 기록·안내하며 재시도와 롤백을 확인한다."""
import heapq
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    s = np.linspace(0, 1, 6)
    risk = 0.0
    for t in threats:
        if np.min(np.hypot(x1 + (x2 - x1) * s - t["x"], y1 + (y2 - y1) * s - t["y"])) < t["r"]:
            risk = max(risk, t["p"])
    return risk


def edge_minutes(e, asset="truck"):
    return e["km"] / (ASSETS[asset]["speed_kmh"] * ROAD_FACTOR[e["road"]]) * 60


def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

nodes, edges = make_network()
threats = make_threats()
DEPOT = "N0000"
for e in edges:
    if e["id"] in ("E017", "E040", "E046", "E071"):
        e["open"] = False
    e["risk"] = edge_risk(e, nodes, threats)
for e in edges:                                   # 가상 고립 지점: N0506 주변 도로 전부 파괴
    if "N0506" in (e["u"], e["v"]):
        e["open"] = False

FLEET = {**{f"T{i:02d}": "truck" for i in range(1, 9)}, **{f"L{i:02d}": "trailer" for i in range(1, 3)}}
LOG = []                                          # 에러·처리 로그(감사용)


class InputError(Exception):
    def __init__(self, code, msg, guide):
        super().__init__(msg); self.code, self.msg, self.guide = code, msg, guide


class TransientError(Exception):
    pass


def validate(req, seen):
    """입력 검증: 필수 필드·형식·범위·중복·존재 여부. 위반 시 InputError(코드, 메시지, 조치 가이드)."""
    for k in ("req_id", "dest", "cargo_t", "deadline_min"):
        if k not in req or req[k] is None:
            raise InputError("V01", f"필수 항목 '{k}' 누락", f"수송 요청서의 '{k}' 칸을 채워 다시 제출")
    rid = str(req["req_id"])
    if not rid.replace("-", "").isalnum() or len(rid) > 20:
        raise InputError("V02", f"요청번호 형식 오류({rid[:12]}...)", "영문·숫자·'-'만 20자 이내로 입력")
    if rid in seen:
        raise InputError("V03", f"중복 요청번호 {rid}", "기존 요청 수정 화면을 사용하거나 새 번호 발급")
    try:
        cargo, dl = float(req["cargo_t"]), float(req["deadline_min"])
    except (TypeError, ValueError):
        raise InputError("V04", f"숫자 형식 오류(cargo_t={req['cargo_t']!r})", "단위 없이 숫자만 입력(예: 12)")
    if not np.isfinite(cargo) or not np.isfinite(dl):
        raise InputError("V05", "적재량·기한 값이 비어 있음(NaN)", "적재량(t)과 도착 기한(분)을 확인")
    if cargo <= 0 or dl <= 0:
        raise InputError("V06", f"음수·0 값(cargo_t={cargo}, deadline={dl})", "0보다 큰 값으로 입력")
    if req["dest"] not in nodes:
        raise InputError("V07", f"존재하지 않는 목적지 {req['dest']}", "도로망 노드 목록에서 목적지 선택")
    cap = sum(ASSETS[k]["cap_t"] for k in FLEET.values())
    if cargo > cap:
        raise InputError("V08", f"적재량 {cargo}t > 가용 수송능력 {cap}t", "분할 수송 요청 또는 상급 제대 지원 요청")
    return cargo, dl


def route(dest, kind, gross, fail_first=False, _state={"n": 0}):
    if fail_first and _state["n"] == 0:          # 경로 계산 서비스 일시 장애 모의(첫 호출만 실패)
        _state["n"] += 1
        raise TransientError("경로 계산 시간 초과")
    adj = build_adj(nodes, edges, lambda e: edge_minutes(e, kind) + 150 * e["risk"] if passable(e, gross) else None)
    cost, path, es = dijkstra(adj, DEPOT, dest)
    if not path:
        raise InputError("R01", f"{dest}까지 통행 가능한 경로 없음", "도로 복구 정보 확인 후 공중수송 검토를 통제관에 요청")
    return path, es


def assign(req, cargo, dl, state, fail_first=False):
    """적재량에 맞게 자산을 고르고(트레일러 우선, 나머지 트럭) 경로를 배정한다. 실패 시 호출자가 롤백."""
    picks, left = [], cargo
    for vid in sorted(FLEET, key=lambda v: FLEET[v] != "trailer"):   # 10t 초과분은 트레일러 우선, 나머지는 트럭
        if left <= 0:
            break
        if vid in state["busy"] or (FLEET[vid] == "trailer" and left <= 10):
            continue
        picks.append(vid); state["busy"].add(vid); left -= ASSETS[FLEET[vid]]["cap_t"]   # 상태 변경(롤백 대상)
    if left > 0:
        raise InputError("A01", "가용 자산 부족", "대기 후 재배정 또는 다른 요청과 통합 수송")
    out = []
    for vid in picks:
        kind = FLEET[vid]
        for attempt in range(3):                  # 일시 장애 자동 재시도(최대 3회)
            try:
                path, es = route(req["dest"], kind, ASSETS[kind]["weight_t"] + min(ASSETS[kind]["cap_t"], cargo), fail_first and attempt == 0)
                break
            except TransientError as ex:
                LOG.append({"req": req["req_id"], "level": "WARN", "code": "T01", "msg": f"{ex} -> 재시도 {attempt + 1}회"})
        mins = sum(edge_minutes(e, kind) for e in es); km = sum(e["km"] for e in es)
        out.append({"req": req["req_id"], "vid": vid, "kind": kind, "dest": req["dest"], "path": path, "km": km, "min": mins,
                    "cost": km * ASSETS[kind]["cost_km"], "threat": sum(e["risk"] > 0 for e in es), "ontime": mins <= dl})
    return out


def process(batch):
    state, seen, result = {"busy": set()}, set(), []
    for req in batch:
        snap = set(state["busy"])                 # 트랜잭션 시작 시점 스냅샷
        try:
            cargo, dl = validate(req, seen)
            rows = assign(req, cargo, dl, state, fail_first=req.get("_flaky", False))
            seen.add(str(req["req_id"])); result += rows
            LOG.append({"req": req["req_id"], "level": "INFO", "code": "OK", "msg": f"배정 {len(rows)}대"})
        except InputError as ex:
            state["busy"] = snap                  # 롤백: 부분 배정한 자산 반환
            LOG.append({"req": str(req.get("req_id"))[:20], "level": "ERROR", "code": ex.code, "msg": ex.msg, "guide": ex.guide})
        except Exception as ex:                   # 예상 못한 오류도 크래시 없이 기록
            state["busy"] = snap
            LOG.append({"req": str(req.get("req_id"))[:20], "level": "ERROR", "code": "X99", "msg": repr(ex)[:60], "guide": "운영 담당자에게 로그 전달"})
    return result, state


# ---------------- 1) 정상 요청 배정 + 관제 대시보드 ----------------
NORMAL = [{"req_id": f"REQ-02{i:02d}", "dest": d, "cargo_t": c, "deadline_min": dl}
          for i, (d, c, dl) in enumerate([("N0307", 12, 90), ("N0107", 8, 80), ("N0407", 4, 70), ("N0205", 9, 75), ("N0503", 5, 60)])]
NORMAL[3]["_flaky"] = True                        # 경로 계산 일시 장애가 나도 재시도로 처리되는지 확인
plan, st = process(NORMAL)
df = pd.DataFrame(plan)
print("[관제 현황판] 요청별 자산 배정·경로 [교육용가상자료]")
summ = df.groupby("req").agg(자산=("vid", lambda s: ",".join(s)), 목적지=("dest", "first"), 거리km=("km", "max"),
                             소요분=("min", "max"), 위협통과=("threat", "max"), 정시=("ontime", "all"), 비용=("cost", "sum")).round(1)
print(summ.to_string())
util = len(st["busy"]) / len(FLEET)
print(f"자산 가동률 {util:.0%} ({len(st['busy'])}/{len(FLEET)}대), 총 운행비용 {df['cost'].sum():.1f}(단가x km), "
      f"정시 도착 예상 {summ['정시'].mean():.0%}, 위협 구역 통과 차량 {int((df['threat'] > 0).sum())}대")

print("일시 장애 자동 재시도:", [f"{l['req']} {l['msg']}" for l in LOG if l["code"] == "T01"], "-> 재시도 후 정상 배정")
print("지연 예상 요청은 붉은색으로 표시하고 통제관에게 기한 조정 또는 자산 변경을 건의한다.")

# ---------------- 2) 예외 주입 테스트 10종 ----------------
LOG.clear()
BAD = [{"dest": "N0307", "cargo_t": 3, "deadline_min": 60},                                   # 1 필수 항목 누락
       {"req_id": "REQ-<script>", "dest": "N0307", "cargo_t": 3, "deadline_min": 60},         # 2 특수문자
       {"req_id": "REQ-0200", "dest": "N0307", "cargo_t": 3, "deadline_min": 60},
       {"req_id": "REQ-0200", "dest": "N0107", "cargo_t": 3, "deadline_min": 60},             # 3 중복 번호
       {"req_id": "REQ-0301", "dest": "N0307", "cargo_t": "12t", "deadline_min": 60},         # 4 단위 포함 문자열
       {"req_id": "REQ-0302", "dest": "N0307", "cargo_t": float("nan"), "deadline_min": 60},  # 5 결측(NaN)
       {"req_id": "REQ-0303", "dest": "N0307", "cargo_t": -4, "deadline_min": 60},            # 6 음수
       {"req_id": "REQ-0304", "dest": "N9999", "cargo_t": 3, "deadline_min": 60},             # 7 없는 목적지
       {"req_id": "REQ-0305", "dest": "N0307", "cargo_t": 500, "deadline_min": 60},           # 8 수송능력 초과
       {"req_id": "REQ-0306", "dest": "N0506", "cargo_t": 3, "deadline_min": 60},             # 9 도달 불가(고립)
       {"req_id": "REQ-0307", "dest": "N0307", "cargo_t": 15, "deadline_min": 60},
       {"req_id": "REQ-0308", "dest": "N0107", "cargo_t": 60, "deadline_min": 60}]            # 10 자산 부족(부분 배정 후 롤백)
expect = ["V01", "V02", "V03", "V04", "V05", "V06", "V07", "V08", "R01", "A01"]
crash = 0
try:
    _, st2 = process(BAD)
except Exception:
    crash += 1
errs = [l for l in LOG if l["level"] == "ERROR"]
print("\n[예외 주입 결과]")
for l in errs:
    print(f"  {l['code']} {l['req']:<14} {l['msg']:<34} -> 조치: {l['guide']}")
caught = [c for c in expect if any(l["code"] == c for l in errs)]
print(f"예외 10종 중 검출 {len(caught)}종, 크래시 {crash}건, 에러 로그 기록률 {len(errs) / len(expect):.0%}")
print(f"롤백 확인: 자산 부족 요청 후 사용 중 자산 {sorted(st2['busy'])} (부분 배정분 반환됨)")

# 대용량: 1만 건 요청을 검증만 수행(경로 계산 제외)
rng = np.random.default_rng(3)
big = [{"req_id": f"B{i:05d}", "dest": rng.choice(list(nodes) + ["N??"]), "cargo_t": float(rng.uniform(-2, 20)), "deadline_min": 60} for i in range(10000)]
t0, seen, nerr = time.perf_counter(), set(), 0
for r in big:
    try:
        validate(r, seen); seen.add(r["req_id"])
    except InputError:
        nerr += 1
print(f"대용량 1만 건 검증 {time.perf_counter() - t0:.2f}초, 오류 판정 {nerr}건, 크래시 0건")
print("감사 로그 예시:", json.dumps(errs[0], ensure_ascii=False))
print(f"목표(크래시 0건, 로그 기록률 100%): {'달성' if crash == 0 and len(caught) == len(expect) else '미달'}")

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for e in edges:
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    ax[0, 0].plot([x1, x2], [y1, y2], color="lightgray" if e["open"] else "black", lw=1, ls="-" if e["open"] else ":")
for t in threats:
    ax[0, 0].add_patch(plt.Circle((t["x"], t["y"]), t["r"], color="red", alpha=0.15))
for i, (rq, g) in enumerate(df.groupby("req")):
    xy = np.array([nodes[n] for n in g.iloc[0]["path"]]); ax[0, 0].plot(xy[:, 0], xy[:, 1], lw=2, label=rq)
ax[0, 0].set_aspect("equal"); ax[0, 0].legend(fontsize=7); ax[0, 0].set_title("Route map (dotted = closed road)")
ax[0, 1].bar(list(FLEET), [1 if v in st["busy"] else 0 for v in FLEET], color=["tab:blue" if FLEET[v] == "truck" else "tab:purple" for v in FLEET])
ax[0, 1].set_title(f"Asset assignment (utilization {util:.0%})"); ax[0, 1].set_yticks([0, 1]); ax[0, 1].set_yticklabels(["idle", "busy"])
for i, r in df.reset_index().iterrows():
    ax[1, 0].barh(r["vid"], r["min"], color="tab:green" if r["ontime"] else "tab:red")
ax[1, 0].set_xlabel("travel minutes"); ax[1, 0].set_title("Convoy schedule (red = late)")
ax[1, 1].bar(summ.index, summ["비용"], color="tab:gray"); ax[1, 1].set_title("Transport cost by request"); ax[1, 1].tick_params(axis="x", rotation=30)
plt.suptitle("MOV02 transport control dashboard (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[관제 현황판] 요청별 자산 배정·경로 [교육용가상자료]
               자산    목적지  거리km    소요분  위협통과     정시     비용
req                                                      
REQ-0200      L01  N0307  60.1  102.0     0  False  144.2
REQ-0201  T01,T02  N0107  42.8   57.4     0   True   85.7
REQ-0202      T03  N0407  54.3   68.6     0   True   54.3
REQ-0203  T04,T05  N0205  35.9   43.1     1   True   71.8
REQ-0204      T06  N0503  40.9   53.4     0   True   40.9
자산 가동률 70% (7/10대), 총 운행비용 396.9(단가x km), 정시 도착 예상 80%, 위협 구역 통과 차량 2대
일시 장애 자동 재시도: ['REQ-0203 경로 계산 시간 초과 -> 재시도 1회'] -> 재시도 후 정상 배정
지연 예상 요청은 붉은색으로 표시하고 통제관에게 기한 조정 또는 자산 변경을 건의한다.

[예외 주입 결과]
  V01 None           필수 항목 'req_id' 누락                  -> 조치: 수송 요청서의 'req_id' 칸을 채워 다시 제출
  V02 REQ-<script>   요청번호 형식 오류(REQ-<script>...)        -> 조치: 영문·숫자·'-'만 20자 이내로 입력
  V03 REQ-0200       중복 요청번호 REQ-0200                   -> 조치: 기존 요청 수정 화면을 사용하거나 새 번호 발급
  V04 REQ-0301       숫자 형식 오류(cargo_t='12t')            -> 조치: 단위 없이 숫자만 입력(예: 12)
  V05 REQ-0302       적재량·기한 값이 비어 있음(NaN)               -> 조치: 적재량(t)과 도착 기한(분)을 확인
  V06 REQ-0303       음수·0 값(cargo_t=-4.0, deadline=60.0) -> 조치: 0보다 큰 값으로 입력
  V07 REQ-0304       존재하지 않는 목적지 N9999                  -> 조치: 도로망 노드 목록에서 목적지 선택
  V08 REQ-0305       적재량 500.0t > 가용 수송능력 80.0t         -> 조치: 분할 수송 요청 또는 상급 제대 지원 요청
  R01 REQ-0306       N0506까지 통행 가능한 경로 없음               -> 조치: 도로 복구 정보 확인 후 공중수송 검토를 통제관에 요청
  A01 REQ-0308       가용 자산 부족                           -> 조치: 대기 후 재배정 또는 다른 요청과 통합 수송
예외 10종 중 검출 10종, 크래시 0건, 에러 로그 기록률 100%
롤백 확인: 자산 부족 요청 후 사용 중 자산 ['L01', 'T01'] (부분 배정분 반환됨)
대용량 1만 건 검증 0.04초, 오류 판정 1167건, 크래시 0건
감사 로그 예시: {"req": "None", "level": "ERROR", "code": "V01", "msg": "필수 항목 'req_id' 누락", "guide": "수송 요청서의 'req_id' 칸을 채워 다시 제출"}
목표(크래시 0건, 로그 기록률 100%): 달성
```
</details>
